In [ ]:
import plotly.express as px
import polars as pl

from fryer import all as fryer

In [ ]:
df_postcode = fryer.data.uk_gov_ons_postcode_directory.read().select(
    pl.col("postcode"),
    pl.col("longitude"),
    pl.col("latitude"),
)
df_postcode.tail().collect()

In [ ]:
df = fryer.data.uk_gov_compare_school_performance.read_raw(year=2023).join(
    df_postcode, on="postcode", how="left"
)

df.head().collect()

In [ ]:
# Postcodes where the join has failed
df.filter(pl.col("longitude").is_null()).collect()["postcode"].value_counts().sort(
    "count", descending=True
)

In [ ]:
for col in (
    "status",
    "group",
    "funding_type",
    "ofsted_rating",
    "religious_character",
):
    df_len = df.group_by([col]).len().collect()
    display(
        df_len.pipe(
            px.bar,
            x=col,
            y="len",
            color=col,
            title=col,
            category_orders={
                col: df_len.sort(by="len", descending=True)[col].to_list(),
            },
        ),
    )